# Required vs. Optional Fields & Mutable Defaults

## 1. Defining Optional Fields with Defaults

By default, any field defined in a Pydantic model with a type hint is **required**. To make a field **optional**, you simply assign it a default value using standard Python assignment syntax (`=`).

```python
from pydantic import BaseModel, ValidationError

class Circle(BaseModel):
    radius: int              # Required field (no default)
    center: tuple[int, int] = (0, 0)  # Optional field (defaults to (0,0))

# We can instantiate without providing 'center'
c1 = Circle(radius=5)
print(c1) # Output: radius=5 center=(0, 0)

```

## 2. The Default Validation "Loophole"

By default, Pydantic **does not validate the default values** you write into your model schema. Pydantic assumes that if you (the developer) hardcode a default, you know what you are doing.

```python
class BrokenCircle(BaseModel):
    radius: int
    # BAD DEFAULT: Type hint is a tuple, but default is a string
    center: tuple[int, int] = "junk" 

# Pydantic will NOT throw an error here, even though "junk" is not a tuple!
c2 = BrokenCircle(radius=5)
print(c2.center) # Output: "junk"

```

*Note: This behavior exists for performance reasons—skipping validation on defaults speeds up model instantiation. You can force validation on defaults via `ConfigDict(validate_default=True)`.*

## 3. The Mutable Default "Gotcha" (And How Pydantic Fixes It)

In standard Python (like in function definitions or standard classes), assigning a **mutable object** (like a `list` or a `dict`) as a default value is a famous trap.

Because Python evaluates default arguments *once* at compile time, all instances would end up sharing the exact same list in memory. If one instance modifies the list, it modifies it for all future instances!

**Standard Python (The Bug):**

```python
def add_item(item, my_list=[]):
    my_list.append(item)
    return my_list

add_item("A") # Returns ["A"]
add_item("B") # Returns ["A", "B"] (Wait, where did "A" come from?!)

```

**Pydantic (The Fix):**
Pydantic is smart enough to handle this automatically. If you define a mutable default in a Pydantic model, Pydantic creates a **deep copy** of that object every time a new instance is created.

```python
class Student(BaseModel):
    name: str
    grades: list[int] = [] # In standard Python, this is a bug. In Pydantic, it's totally safe!

s1 = Student(name="Alice")
s1.grades.append(100)

s2 = Student(name="Bob")
print(s2.grades) # Output: [] (Bob starts with an empty list, Alice's data did not bleed over!)

```

*(Note: While Pydantic allows this, standard Python Dataclasses do not. In Dataclasses, you would be forced to use a `default_factory` to achieve this behavior).*

In [4]:
from pydantic import BaseModel, ValidationError

class Circle(BaseModel):
    center : tuple[int,int] = (0,0)
    radius : int

In [5]:
c1 =Circle(radius = 2)
print(c1.center, type(c1.center))
print(c1.radius)

(0, 0) <class 'tuple'>
2


In [7]:
c2 =Circle(center =(1,1), radius = 2)
print(c2.center, type(c2.center))
print(c2.radius)

(1, 1) <class 'tuple'>
2


In [9]:
try :
    Circle(center =(1,1))
except ValidationError as e:
    print(e)


1 validation error for Circle
radius
  Field required [type=missing, input_value={'center': (1, 1)}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [ ]:
def add_item(item, my_list=["C"]):
    my_list.append(item)
    return my_list

print(add_item("A")) # Returns ["A"]
add_item("B") # Returns ["A", "B"] (Wait, where did "A" come from?!)

### Pydantic Optional Fields & Defaults: Interview Prep

<details>
<summary><b>1. How do you make a field optional in a Pydantic model, and what happens when that field is omitted during deserialization?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
You make a field optional simply by providing it with a default value in the model definition, exactly like providing a default argument in a standard Python function (e.g., `center: tuple[int, int] = (0, 0)`). 

When data is deserialized into the model, if the incoming payload is missing that specific field, Pydantic will automatically populate the model instance with the defined default value. It no longer raises a `ValidationError` for a missing required field.
</details>

<details>
<summary><b>2. If you assign a default value of the wrong type to a field (e.g., `radius: int = "junk"`), when does Pydantic catch this error by default?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
By default, Pydantic *does not* catch this error. 

Pydantic assumes that as the developer writing the class definition, you are providing logically sound defaults. It skips validation on the default values to save compute time and optimize performance. Therefore, if you omit the `radius` during instantiation, the model will successfully initialize with the string `"junk"` as the radius. (Though you can force Pydantic to validate defaults via `validate_default=True` in the model's Field configuration).
</details>

<details>
<summary><b>3. In standard Python, defining a mutable object (like a list or dict) as a default function argument is a dangerous anti-pattern. Why is this, and how does Pydantic handle mutable defaults differently in `BaseModel`?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
In standard Python, default arguments are evaluated exactly *once* when the function (or class) is compiled. If you set a default to `my_list = []`, that exact single list object is stored in memory. Every time the function is called without that argument, it shares that *same* list. If you append to it, the list grows across all future function calls, leading to massive bugs.

In standard Python `dataclasses`, you fix this using `default_factory`. 

However, Pydantic goes a step further. If you define a mutable default directly in a Pydantic model (e.g., `tags: list[str] = []`), Pydantic intercepts this under the hood. Every time a new instance of the model is created, Pydantic automatically performs a **deep copy** of that mutable default. This ensures that every model instance gets its own isolated list, making it perfectly safe to write `tags: list = []` in Pydantic.
</details>

<details>
<summary><b>4. A Junior developer writes `tags: list[str] = None` to make a field optional. What is the semantic difference between that and writing `tags: list[str] = []` in Pydantic?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
While both make the field optional, they lead to entirely different downstream handling. 

If they use `= None`, the default state of the field is exactly `None`. Downstream code must constantly perform `if user.tags is not None:` checks before trying to iterate or append to the tags, otherwise, they will get a `TypeError`.

If they use `= []`, the default state is an empty list. Downstream code can safely assume `user.tags` is always an iterable list. You can safely call `for tag in user.tags:` or `user.tags.append("new")` without writing defensive `None` checks, leading to much cleaner code.
</details>

<details>
<summary><b>5. If you have an optional field but you *must* calculate the default value dynamically at runtime (e.g., setting a `created_at` timestamp), how do you do this in Pydantic?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
You cannot assign a dynamic function call directly to the attribute (e.g., `created_at: datetime = datetime.now()`). If you do, `datetime.now()` executes only once when the module is imported, meaning every single instance created during that application lifecycle will share the exact same timestamp.

Instead, you use Pydantic's `Field` with a `default_factory`. 
You define it as: `created_at: datetime = Field(default_factory=datetime.now)`. This tells Pydantic to execute the `datetime.now` callable fresh every single time a new instance is instantiated.
</details>


# Nullable Fields & The Combinations Matrix

## 1. Optional vs. Nullable (The Big Disambiguation)

In standard conversation, we use these words interchangeably. In Pydantic, they have strict, orthogonal definitions:

* **Optional:** Dictates **Payload Presence**. A field is optional *only* if it has a default value (e.g., `= 0`). The client does not need to send this key in the JSON.
* **Nullable:** Dictates **Type Flexibility**. A field is nullable *only* if the type hint explicitly accepts `None` (e.g., `int | None`). The client is allowed to send the JSON `null` object.

## 2. Syntax for Nullable Fields

To define a field that accepts `None`, you must modify the type hint.

* **Modern (Python 3.10+):** `field: int | None` *(Best Practice)*
* **Legacy (Python < 3.10):** `field: Union[int, None]`
* **The Trap:** `field: Optional[int]`

> **⚠️ Why `typing.Optional` is an Anti-Pattern in Pydantic:** > The name `Optional[int]` implies the field doesn't need to be provided. But to Pydantic, a type hint of `Optional[int]` without a default value means the field is **Required**. The client *must* pass the key in the payload, even if the value is `null`. Because the nomenclature is so contradictory, modern Python and Pydantic V2 strongly favor the `int | None` syntax.

## 3. The Validation "Blind Spot" Anti-Pattern

Beginners constantly write this:

```python
class BadModel(BaseModel):
    field: int = None  # ❌ DANGEROUS

```

**Why does this work without an error?** Pydantic explicitly *does not validate default values* at runtime to save compute cycles. It trusts the developer.
**Why is it dangerous?** Downstream code (and your IDE/MyPy) will assume `field` is always an integer. When the model initializes with `None`, and you try to do math on it, your application will crash. **Always align the type hint with the default: `field: int | None = None`.**

---

## 4. The Four Combinations Matrix

By combining these two properties, we get four exact behaviors for how an API endpoint will handle incoming JSON.

### A. Required, Not Nullable

* **Syntax:** `field: int`
* **Behavior:** The JSON payload *must* contain the key, and the value *must* be an integer.
* **If Omitted:** ❌ `ValidationError` (Field required)
* **If `null`:** ❌ `ValidationError` (Input should be a valid integer)

### B. Required, Nullable

* **Syntax:** `field: int | None`
* **Behavior:** The JSON payload *must* contain the key, but the client is allowed to explicitly pass `null`.
* **If Omitted:** ❌ `ValidationError` (Field required)
* **If `null`:** ✅ Success (`field=None`)

### C. Optional, Not Nullable

* **Syntax:** `field: int = 0`
* **Behavior:** The client can omit the key (it defaults to 0). But if they *do* send the key, it cannot be `null`.
* **If Omitted:** ✅ Success (`field=0`)
* **If `null`:** ❌ `ValidationError` (Input should be a valid integer)

### D. Optional, Nullable

* **Syntax:** `field: int | None = None`
* **Behavior:** The most flexible. The client can omit the key, or explicitly pass `null`.
* **If Omitted:** ✅ Success (`field=None`)
* **If `null`:** ✅ Success (`field=None`)



### Interview Preparation: Nullable vs. Optional

<details>
<summary><b>1. What is the fundamental difference between an "Optional" field and a "Nullable" field in Pydantic?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
An Optional field dictates payload presence. If a field has a default value (e.g., `age: int = 25`), it is optional because the client does not have to send that key in the JSON request. <br><br>
A Nullable field dictates type constraints. If a field type hint includes `None` (e.g., `age: int | None`), it is nullable, meaning the system explicitly accepts the `null` object. A field can be required but nullable (must send the key, but can be `null`), or optional but not nullable (can skip the key, but if sent, cannot be `null`).
</details>

<details>
<summary><b>2. Why do senior developers prefer `int | None` over `typing.Optional[int]` in Pydantic models?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
Because `typing.Optional` creates severe cognitive dissonance. In standard English, "optional" means you don't have to provide it. But in Pydantic, a field defined as `my_field: Optional[int]` (without a default value) is actually a **strictly required** field. The client must include it in the JSON. <br><br>
`int | None` clearly communicates exactly what the constraint is: "This value must be an integer, or it must be None." It separates the concept of type from the concept of payload requirement.
</details>

<details>
<summary><b>3. You see the code `retry_count: int = None` in a PR. What is the architectural danger of this line, and why doesn't Pydantic throw an error immediately?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
This is a massive anti-pattern. The developer assigned a default of `None` to make the field optional, but failed to make the type hint nullable (`int | None`). <br><br>
Pydantic does not throw an error during instantiation because, for performance reasons, it trusts the developer and skips validation on default values. However, downstream code, static type checkers (MyPy), and IDEs will assume `retry_count` is guaranteed to be an integer. When the app runs and tries to execute `retry_count + 1`, the app will hit a fatal `TypeError` because you cannot add an integer to `None`.
</details>

<details>
<summary><b>4. You are building a FastAPI endpoint to update a user's database profile via a PATCH request. The user should be able to clear their existing "bio" by passing `null`, or leave the existing "bio" unchanged by omitting the key entirely. How do you construct this field?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
This scenario requires the field to be *both* Optional and Nullable. I would define it as `bio: str | None = None`. <br><br>
When I receive the request, I need to know if `bio` was omitted, or if they explicitly passed `null`. To extract the data for the database query, I call `model.model_dump(exclude_unset=True)`. <br><br>
If they omitted the key, `bio` won't be in the exported dictionary, and my SQL query leaves the column alone. If they passed `{"bio": null}`, Pydantic tracks that the field was explicitly "set", so it will be included in the dump as `{"bio": None}`, allowing me to dynamically write a SQL query that nullifies the column.
</details>

# Inspecting Fields & Explicit Assignments

When working with Pydantic, data originates from two places: the explicit payload (what the user sent) and the implicit defaults (what your model injected). Inspecting fields allows you to differentiate between the two.

## 1. Inspecting the Class Definition (`model_fields`)

Before a model is even instantiated, you can inspect its metadata by accessing the `model_fields` class attribute. This returns a dictionary mapping field names to `FieldInfo` objects.

```python
from pydantic import BaseModel

class Circle(BaseModel):
    center_x: int = 0
    center_y: int = 0
    radius: int
    name: str | None = None

# Inspecting the model class definition
print(Circle.model_fields.keys())
# dict_keys(['center_x', 'center_y', 'radius', 'name'])

# Inspecting a specific field's metadata
print(Circle.model_fields['center_x'].is_required()) # False
print(Circle.model_fields['center_x'].default)       # 0

```

*Use Case:* Automatically generating documentation, building dynamic forms, or creating generic validation functions that inspect model schemas at runtime.

## 2. Inspecting an Instance (`model_fields_set`)

Once you instantiate a model, you often need to know *exactly* which fields were provided in the incoming data. You do this using the `model_fields_set` property on the instance.

```python
# The user only provides the radius
c1 = Circle(radius=5)

# Pydantic populates the rest with defaults
print(c1.model_dump())
# {'center_x': 0, 'center_y': 0, 'radius': 5, 'name': None}

# Which fields were EXPLICITLY set by the user?
print(c1.model_fields_set)
# {'radius'}

```

### Deriving the "Defaulted" Fields

Pydantic does not have a built-in `model_fields_defaulted` property. However, because both `model_fields` and `model_fields_set` can be treated as Python Sets, you can find the defaulted fields using a simple set difference:

```python
all_fields = set(c1.model_fields.keys())
explicit_fields = c1.model_fields_set

defaulted_fields = all_fields - explicit_fields
print(defaulted_fields) 
# {'center_x', 'center_y', 'name'}

```

## 3. Practical API Application: Echoing & Updating

If a user sends a `PATCH` request with only one field, and you return the fully serialized model, they might be confused to see three other default fields magically appended to their data.

To return (or database-update) *only* what the user explicitly provided, we leverage this "set" state.

**Method A: Manual Inclusion (Transcript Method)**
You can dynamically pass the `model_fields_set` directly to the `include` parameter during serialization.

```python
user_payload = {"radius": 10}
c2 = Circle.model_validate(user_payload)

# Dump ONLY what the user sent
echo_data = c2.model_dump(include=c2.model_fields_set)
print(echo_data) # {'radius': 10}

```

**Method B: Built-in Exclusion (The Senior / Pythonic Method)**
Pydantic actually has a built-in parameter specifically designed to handle this exact scenario under the hood. Using `exclude_unset=True` achieves the exact same result with cleaner, more idiomatic code.

```python
# The industry standard way to do the exact same thing:
echo_data = c2.model_dump(exclude_unset=True)
print(echo_data) # {'radius': 10}

```

In [1]:
from pydantic import BaseModel, ValidationError


In [2]:
class circle(BaseModel):
    center_x : int = 0
    center_y : int = 0
    radius : int = 1
    name : str|None = None

In [3]:
circle.model_fields

{'center_x': FieldInfo(annotation=int, required=False, default=0),
 'center_y': FieldInfo(annotation=int, required=False, default=0),
 'radius': FieldInfo(annotation=int, required=False, default=1),
 'name': FieldInfo(annotation=Union[str, NoneType], required=False, default=None)}

In [4]:
c1 = circle(radius=2)
c2= circle(name ="circle2")
c3 = circle()

In [5]:
c1

circle(center_x=0, center_y=0, radius=2, name=None)

In [6]:
c2

circle(center_x=0, center_y=0, radius=1, name='circle2')

In [7]:
c3

circle(center_x=0, center_y=0, radius=1, name=None)

In [8]:
c1.model_fields_set

{'radius'}

In [9]:
c2.model_fields_set

{'name'}

In [10]:
c3.model_fields_set

set()

In [14]:
# lets find the default valued keys set
c1_all_fields = set(circle.model_fields.keys())
c1_set_set = c1.model_fields_set
c1_default_fields = c1_all_fields - c1_set_set

In [15]:
c1_default_fields

{'center_x', 'center_y', 'name'}

### Interview Preparation: Inspecting Fields & Explicit Assignments

<details>
<summary><b>1. What is the difference between `model_fields` and `model_fields_set` in Pydantic?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
`model_fields` is a class-level attribute. It contains the metadata and schema definition for the model (types, defaults, aliases, required status) before any data is ever loaded into it. <br><br>
`model_fields_set` is an instance-level property. It returns a Python `set` of strings representing only the field names that were explicitly provided during the instantiation or deserialization of that specific object.
</details>

<details>
<summary><b>2. If a user explicitly sends `{"status": "active"}` in a JSON payload, but the Pydantic model's default for status is *also* `"active"`, does it show up in `model_fields_set`?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
Yes. `model_fields_set` tracks <i>payload presence</i>, not value deviation. Because the user explicitly included the `"status"` key in their JSON payload, Pydantic records it as "set," even if the value they provided perfectly matches the fallback default value.
</details>

<details>
<summary><b>3. How do you programmatically determine which fields in a Pydantic instance were populated by default values instead of explicit user input?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
Pydantic does not track "defaulted" fields directly, but because we can extract the model keys and the set fields as Python sets, we can use a simple set difference operation. <br><br>
You can subtract the instance's set fields from the model's total fields: <br>
`defaulted_fields = set(instance.model_fields.keys()) - instance.model_fields_set`.
</details>

<details>
<summary><b>4. You are building an API endpoint where users can partially update their profile. They send a JSON payload containing only the fields they want to change. Why is `exclude_unset=True` critical when serializing this data to your database ORM?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
If I use a standard `model_dump()`, Pydantic will serialize the fields the user sent, *plus* the default values of all the fields they omitted. If I pass that full dictionary to my ORM, I will accidentally overwrite their existing database records with default values. <br><br>
By using `model_dump(exclude_unset=True)`, Pydantic uses `model_fields_set` under the hood to serialize *only* the data explicitly present in the incoming payload. This allows me to pass a clean, partial dictionary to my ORM, ensuring I only update the exact columns the user requested to change.
</details>


# JSON Schema Generation

## 1. What is a JSON Schema?

A JSON Schema is a standardized vocabulary that allows you to annotate and validate JSON documents. In the context of API development (like REST APIs), JSON Schemas are the underlying engine that power tools like Swagger/OpenAPI. They describe exactly what an API endpoint expects to receive and exactly what it promises to return.

Writing these schemas by hand is incredibly tedious and error-prone. Fortunately, Pydantic generates them automatically directly from your Python model definitions.

## 2. Generating the Schema in Pydantic

You can extract the JSON schema representation of any Pydantic model using the `model_json_schema()` method. This returns a standard Python dictionary representing the schema.

```python
from pydantic import BaseModel
from pprint import pprint

class Model(BaseModel):
    # Nullable and Optional
    field_1: int | None = None
    
    # Optional, but Not Nullable
    field_2: str = "Python"

# Generate and print the schema
pprint(Model.model_json_schema())

```

### The Output Breakdown:

```python
{'properties': {'field_1': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
                            'default': None,
                            'title': 'Field 1'},
                'field_2': {'default': 'Python',
                            'title': 'Field 2',
                            'type': 'string'}},
 'title': 'Model',
 'type': 'object'}

```

* **`title` & `type`:** Pydantic automatically infers the model name (`Model`) and labels it as a JSON `object`.
* **`field_1`:** Because we used `int | None`, the schema uses the `anyOf` keyword, indicating the value can be either an integer or null.
* **`field_2`:** Because it is just a string, it simply lists `'type': 'string'`. Notice how Pydantic automatically creates human-readable titles (e.g., `'Field 2'`) out of your snake_case variable names.

## 3. Real-World Application (FastAPI & LangChain)

* **FastAPI:** You almost never have to call `model_json_schema()` manually in FastAPI. When you define a route, FastAPI reads the Pydantic model, calls this exact method under the hood, and injects the output into the OpenAPI specification to generate your interactive documentation.
* **LangChain / OpenAI API:** When you want to force an LLM to return structured data (using OpenAI's "Function Calling" or LangChain's `with_structured_output`), the framework converts your Pydantic model into a JSON Schema and sends *that schema* to the LLM so it knows exactly what keys and types to return.